In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)
 
print("=" * 70)
print("  CHAPITRE 3 — DT-HMM : Détection d'Intrusions (NSL-KDD)")
print("  Modèles de Markov Cachés à Temps Discret")
print("=" * 70)

  CHAPITRE 3 — DT-HMM : Détection d'Intrusions (NSL-KDD)
  Modèles de Markov Cachés à Temps Discret


#  PARTIE 0 : Chargement NSL-KDD


In [2]:

COLONNES = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root',
    'num_file_creations','num_shells','num_access_files','num_outbound_cmds',
    'is_host_login','is_guest_login','count','srv_count','serror_rate',
    'srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate',
    'diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
    'dst_host_serror_rate','dst_host_srv_serror_rate','dst_host_rerror_rate',
    'dst_host_srv_rerror_rate','label','difficulty'
]
ATTACK_MAP = {
    'normal':'normal',
    'back':'dos','land':'dos','neptune':'dos','pod':'dos','smurf':'dos',
    'teardrop':'dos','apache2':'dos','udpstorm':'dos','processtable':'dos',
    'ipsweep':'probe','nmap':'probe','portsweep':'probe','satan':'probe',
    'mscan':'probe','saint':'probe',
    'ftp_write':'r2l','guess_passwd':'r2l','imap':'r2l','multihop':'r2l',
    'phf':'r2l','warezclient':'r2l','warezmaster':'r2l',
    'buffer_overflow':'u2r','loadmodule':'u2r','perl':'u2r','rootkit':'u2r'
}
 
CHEMIN = r'C:\Users\Fujitsu\Desktop\Data science\S3\ML probabilistique\project of module\projet_cybersecurity\dataset\KDDTrain+.txt'
 
try:
    df = pd.read_csv(CHEMIN, header=None, names=COLONNES)
    df['attack_cat'] = df['label'].map(ATTACK_MAP).fillna('other')
    df = df.sample(n=min(8000, len(df)), random_state=42).reset_index(drop=True)
    print(f"✅ NSL-KDD chargé : {len(df)} connexions")
except:
    # Dataset synthétique réaliste
    n = 8000
    cats = ['normal']*4000 + ['dos']*2000 + ['probe']*1200 + ['r2l']*500 + ['u2r']*300
    np.random.shuffle(cats)
    cats = cats[:n]
    rows = []
    for cat in cats:
        if cat == 'normal':
            row = {'serror_rate':np.random.uniform(0,0.1),
                   'rerror_rate':np.random.uniform(0,0.1),
                   'src_bytes':np.random.randint(100,5000),
                   'num_failed_logins':np.random.choice([0,1],p=[0.97,0.03]),
                   'flag':np.random.choice(['SF','S0'],p=[0.95,0.05]),
                   'attack_cat':cat}
        elif cat == 'dos':
            row = {'serror_rate':np.random.uniform(0.6,1.0),
                   'rerror_rate':np.random.uniform(0,0.2),
                   'src_bytes':np.random.randint(0,100),
                   'num_failed_logins':0,
                   'flag':np.random.choice(['S0','S1'],p=[0.8,0.2]),
                   'attack_cat':cat}
        elif cat == 'probe':
            row = {'serror_rate':np.random.uniform(0.1,0.6),
                   'rerror_rate':np.random.uniform(0.2,0.8),
                   'src_bytes':np.random.randint(0,500),
                   'num_failed_logins':np.random.choice([0,1],p=[0.7,0.3]),
                   'flag':np.random.choice(['SF','REJ'],p=[0.5,0.5]),
                   'attack_cat':cat}
        else:
            row = {'serror_rate':np.random.uniform(0,0.3),
                   'rerror_rate':np.random.uniform(0,0.3),
                   'src_bytes':np.random.randint(0,1000),
                   'num_failed_logins':np.random.choice([0,1,2],p=[0.5,0.3,0.2]),
                   'flag':np.random.choice(['SF','REJ'],p=[0.6,0.4]),
                   'attack_cat':cat}
        rows.append(row)
    df = pd.DataFrame(rows)
    print(f"✅ NSL-KDD simulé : {len(df)} connexions")
 
print(f"\n  Distribution :")
for cat, cnt in df['attack_cat'].value_counts().items():
    print(f"    {cat:10s} : {cnt:5d}  ({cnt/len(df)*100:.1f}%)")

✅ NSL-KDD simulé : 8000 connexions

  Distribution :
    normal     :  4000  (50.0%)
    dos        :  2000  (25.0%)
    probe      :  1200  (15.0%)
    r2l        :   500  (6.2%)
    u2r        :   300  (3.8%)


#  PARTIE 1 : Introduction HMM — Définition des états cachés et observations


In [3]:
print("\n" + "=" * 70)
print("PARTIE 1 — Introduction aux HMM : États Cachés & Observations")
print("=" * 70)
 
# ── États cachés S (non observables directement) ──
# Comme dans le cours : TN=Transaction Normale → ici : Normal, Intrusion
ETATS_CACHES = {0: "Normal", 1: "Intrusion"}
N = len(ETATS_CACHES)   # nombre d'états cachés
 
# ── Observations O (ce qu'on observe dans NSL-KDD) ──
# Inspiré du cours : Faible/Moyen/Élevé → ici : trafic Faible/Moyen/Élevé
# Basé sur serror_rate + src_bytes discrétisé
OBS_LABELS = {0: "Trafic_Faible", 1: "Trafic_Moyen", 2: "Trafic_Eleve"}
M = len(OBS_LABELS)     # nombre d'observations possibles
 
def discretiser_observation(row):
    """Discrétise les features NSL-KDD en 3 niveaux d'observation."""
    serr = row.get('serror_rate', 0)
    fails = row.get('num_failed_logins', 0)
    if serr < 0.2 and fails == 0:
        return 0   # Trafic Faible
    elif serr < 0.6 and fails <= 1:
        return 1   # Trafic Moyen
    else:
        return 2   # Trafic Élevé
 
df['observation'] = df.apply(discretiser_observation, axis=1)
df['etat_cache'] = df['attack_cat'].apply(lambda x: 0 if x == 'normal' else 1)
 
print(f"""
  Analogie avec le cours (Détection de Fraude → Détection d'Intrusions) :
  ┌─────────────────────────┬──────────────────────────────────────┐
  │ Cours (Fraude)          │ Notre projet (NSL-KDD)               │
  ├─────────────────────────┼──────────────────────────────────────┤
  │ Trans. Normale (TN)     │ État Normal (trafic légitime)        │
  │ Trans. Frauduleuse (TF) │ État Intrusion (attaque réseau)      │
  │ Montant Faible (MF)     │ Trafic Faible (serror<0.2, fails=0)  │
  │ Montant Moyen (MM)      │ Trafic Moyen  (serror<0.6)           │
  │ Montant Élevé (ME)      │ Trafic Élevé  (serror≥0.6)           │
  └─────────────────────────┴──────────────────────────────────────┘
 
  États cachés S = {{S0: Normal, S1: Intrusion}}   → N = {N}
  Observations  O = {{O0: Trafic_Faible, O1: Trafic_Moyen, O2: Trafic_Eleve}} → M = {M}
""")
 
print(f"  Distribution des observations dans NSL-KDD :")
for k, lbl in OBS_LABELS.items():
    cnt = (df['observation']==k).sum()
    print(f"    O{k} ({lbl:16s}) : {cnt:5d}  ({cnt/len(df)*100:.1f}%)")


PARTIE 1 — Introduction aux HMM : États Cachés & Observations

  Analogie avec le cours (Détection de Fraude → Détection d'Intrusions) :
  ┌─────────────────────────┬──────────────────────────────────────┐
  │ Cours (Fraude)          │ Notre projet (NSL-KDD)               │
  ├─────────────────────────┼──────────────────────────────────────┤
  │ Trans. Normale (TN)     │ État Normal (trafic légitime)        │
  │ Trans. Frauduleuse (TF) │ État Intrusion (attaque réseau)      │
  │ Montant Faible (MF)     │ Trafic Faible (serror<0.2, fails=0)  │
  │ Montant Moyen (MM)      │ Trafic Moyen  (serror<0.6)           │
  │ Montant Élevé (ME)      │ Trafic Élevé  (serror≥0.6)           │
  └─────────────────────────┴──────────────────────────────────────┘
 
  États cachés S = {S0: Normal, S1: Intrusion}   → N = 2
  Observations  O = {O0: Trafic_Faible, O1: Trafic_Moyen, O2: Trafic_Eleve} → M = 3

  Distribution des observations dans NSL-KDD :
    O0 (Trafic_Faible   ) :  4297  (53.7%)
    O1 

#  PARTIE 2 : Paramètres du HMM λ = (A, B, π) depuis NSL-KDD


In [4]:
print("\n" + "=" * 70)
print("PARTIE 2 — Paramètres HMM λ = (A, B, π) estimés depuis NSL-KDD")
print("=" * 70)
 
# ── Matrice de transition A (N×N) ──
# a_ij = P(X_{t+1}=j | X_t=i)
counts_A = np.zeros((N, N))
etats = df['etat_cache'].values
for t in range(len(etats)-1):
    i, j = int(etats[t]), int(etats[t+1])
    counts_A[i, j] += 1
 
A = counts_A / counts_A.sum(axis=1, keepdims=True)
 
# ── Matrice d'émission B (N×M) ──
# b_ij = P(O_t=j | X_t=i)
counts_B = np.zeros((N, M))
for idx, row in df.iterrows():
    i = int(row['etat_cache'])
    k = int(row['observation'])
    counts_B[i, k] += 1
 
B = counts_B / counts_B.sum(axis=1, keepdims=True)
 
# ── Distribution initiale π ──
pi = np.array([
    (df['etat_cache']==0).sum() / len(df),
    (df['etat_cache']==1).sum() / len(df)
])
 
print(f"""
  Matrice de transition A (a_ij = P(X_t+1=j | X_t=i)) :
         Normal   Intrusion
  Normal  {A[0,0]:.4f}    {A[0,1]:.4f}
  Intrus  {A[1,0]:.4f}    {A[1,1]:.4f}
 
  → Une connexion normale reste normale avec {A[0,0]*100:.1f}% de probabilité
  → Une intrusion reste intrusion avec {A[1,1]*100:.1f}% de probabilité
 
  Matrice d'émission B (b_ik = P(O_t=k | X_t=i)) :
         Traf_Faible  Traf_Moyen  Traf_Eleve
  Normal  {B[0,0]:.4f}      {B[0,1]:.4f}      {B[0,2]:.4f}
  Intrus  {B[1,0]:.4f}      {B[1,1]:.4f}      {B[1,2]:.4f}
 
  → En cas d'intrusion, P(Trafic_Élevé) = {B[1,2]*100:.1f}%
    (analogue au cours : en cas de fraude, P(Montant_Élevé) = 60%)
 
  Distribution initiale π :
    π(Normal)    = {pi[0]:.4f}  ({pi[0]*100:.1f}%)
    π(Intrusion) = {pi[1]:.4f}  ({pi[1]*100:.1f}%)
""")
 
print(f"  Hypothèse de Markov (1er ordre) :")
print(f"    P(X_i | X_1,...,X_i-1) = P(X_i | X_i-1)")
print(f"  Indépendance des sorties :")
print(f"    P(O_i | X_1,...,X_T, O_1,...,O_T) = P(O_i | X_i)")


PARTIE 2 — Paramètres HMM λ = (A, B, π) estimés depuis NSL-KDD

  Matrice de transition A (a_ij = P(X_t+1=j | X_t=i)) :
         Normal   Intrusion
  Normal  0.5035    0.4965
  Intrus  0.4964    0.5036
 
  → Une connexion normale reste normale avec 50.3% de probabilité
  → Une intrusion reste intrusion avec 50.4% de probabilité
 
  Matrice d'émission B (b_ik = P(O_t=k | X_t=i)) :
         Traf_Faible  Traf_Moyen  Traf_Eleve
  Normal  0.9647      0.0352      0.0000
  Intrus  0.1095      0.3530      0.5375
 
  → En cas d'intrusion, P(Trafic_Élevé) = 53.8%
    (analogue au cours : en cas de fraude, P(Montant_Élevé) = 60%)
 
  Distribution initiale π :
    π(Normal)    = 0.5000  (50.0%)
    π(Intrusion) = 0.5000  (50.0%)

  Hypothèse de Markov (1er ordre) :
    P(X_i | X_1,...,X_i-1) = P(X_i | X_i-1)
  Indépendance des sorties :
    P(O_i | X_1,...,X_T, O_1,...,O_T) = P(O_i | X_i)


#  PARTIE 3 : Algorithme de Forward α


In [5]:
print("\n" + "=" * 70)
print("PARTIE 3 — Algorithme de Forward α_t(i)")
print("=" * 70)
print("""
  Définition : α_t(i) = P(O_1,...,O_t, X_t=i | λ)
  = probabilité d'observer O_1..O_t ET d'être en état i à l'instant t
 
  Initialisation : α_1(i) = π_i · B_i(O_1)
  Récurrence    : α_{t+1}(j) = [Σ_i α_t(i) · a_ij] · B_j(O_{t+1})
  Finalisation  : P(O|λ) = Σ_i α_T(i)
""")
 
def forward(obs_seq, A, B, pi):
    """
    Algorithme de Forward.
    obs_seq : séquence d'indices d'observations (0..M-1)
    Retourne alpha (T×N) et P(O|λ)
    """
    T = len(obs_seq)
    N = A.shape[0]
    alpha = np.zeros((T, N))
 
    # Initialisation
    alpha[0] = pi * B[:, obs_seq[0]]
 
    # Récurrence
    for t in range(1, T):
        for j in range(N):
            alpha[t, j] = np.sum(alpha[t-1] * A[:, j]) * B[j, obs_seq[t]]
 
    # Finalisation
    prob_obs = alpha[-1].sum()
    return alpha, prob_obs
 
# Séquence de test : 10 premières observations de NSL-KDD
obs_seq_test = df['observation'].values[:10]
etats_reels  = df['etat_cache'].values[:10]
 
alpha, prob_forward = forward(obs_seq_test, A, B, pi)
 
print(f"  Séquence d'observations (10 premières connexions NSL-KDD) :")
print(f"    O = {[OBS_LABELS[o] for o in obs_seq_test]}")
print(f"  États réels (cachés) :")
print(f"    X = {[ETATS_CACHES[e] for e in etats_reels]}")
print(f"\n  Matrice α (T×N) — probabilités forward :")
print(f"  {'t':>4s}  {'α_t(Normal)':>16s}  {'α_t(Intrusion)':>16s}")
for t in range(len(obs_seq_test)):
    print(f"  {t+1:>4d}  {alpha[t,0]:>16.8f}  {alpha[t,1]:>16.8f}")
print(f"\n  P(O|λ) = Σ_i α_T(i) = {prob_forward:.10f}")
print(f"\n  → Interprétation : La probabilité que cette séquence de {len(obs_seq_test)}")
print(f"    connexions soit générée par notre modèle HMM est {prob_forward:.2e}")


PARTIE 3 — Algorithme de Forward α_t(i)

  Définition : α_t(i) = P(O_1,...,O_t, X_t=i | λ)
  = probabilité d'observer O_1..O_t ET d'être en état i à l'instant t
 
  Initialisation : α_1(i) = π_i · B_i(O_1)
  Récurrence    : α_{t+1}(j) = [Σ_i α_t(i) · a_ij] · B_j(O_{t+1})
  Finalisation  : P(O|λ) = Σ_i α_T(i)

  Séquence d'observations (10 premières connexions NSL-KDD) :
    O = ['Trafic_Faible', 'Trafic_Faible', 'Trafic_Faible', 'Trafic_Moyen', 'Trafic_Eleve', 'Trafic_Faible', 'Trafic_Faible', 'Trafic_Faible', 'Trafic_Eleve', 'Trafic_Faible']
  États réels (cachés) :
    X = ['Normal', 'Normal', 'Normal', 'Normal', 'Intrusion', 'Normal', 'Normal', 'Normal', 'Intrusion', 'Normal']

  Matrice α (T×N) — probabilités forward :
     t       α_t(Normal)    α_t(Intrusion)
     1        0.48237500        0.05475000
     2        0.26053295        0.02924446
     3        0.14055880        0.01577708
     4        0.00277074        0.02743982
     5        0.00000000        0.00816735
     6  

#  PARTIE 4 : Algorithme de Backward β


In [6]:
print("\n" + "=" * 70)
print("PARTIE 4 — Algorithme de Backward β_t(i)")
print("=" * 70)
print("""
  Définition : β_t(i) = P(O_{t+1},...,O_T | X_t=i, λ)
  = probabilité d'observer la suite O_{t+1}..O_T en partant de l'état i à t
 
  Initialisation : β_T(i) = 1 pour tout i
  Récurrence    : β_t(i) = Σ_j a_ij · B_j(O_{t+1}) · β_{t+1}(j)
  Finalisation  : P(O|λ) = Σ_i π_i · B_i(O_1) · β_1(i)
""")
 
def backward(obs_seq, A, B, pi):
    """Algorithme de Backward."""
    T = len(obs_seq)
    N = A.shape[0]
    beta = np.zeros((T, N))
 
    # Initialisation
    beta[-1] = 1.0
 
    # Récurrence (de T-2 à 0)
    for t in range(T-2, -1, -1):
        for i in range(N):
            beta[t, i] = np.sum(A[i, :] * B[:, obs_seq[t+1]] * beta[t+1, :])
 
    # Finalisation
    prob_obs = np.sum(pi * B[:, obs_seq[0]] * beta[0, :])
    return beta, prob_obs
 
beta, prob_backward = backward(obs_seq_test, A, B, pi)
 
print(f"  Matrice β (T×N) — probabilités backward :")
print(f"  {'t':>4s}  {'β_t(Normal)':>16s}  {'β_t(Intrusion)':>16s}")
for t in range(len(obs_seq_test)):
    print(f"  {t+1:>4d}  {beta[t,0]:>16.8f}  {beta[t,1]:>16.8f}")
print(f"\n  P(O|λ) via backward = {prob_backward:.10f}")
print(f"  Cohérence Forward = Backward : {np.isclose(prob_forward, prob_backward, rtol=1e-5)}")


PARTIE 4 — Algorithme de Backward β_t(i)

  Définition : β_t(i) = P(O_{t+1},...,O_T | X_t=i, λ)
  = probabilité d'observer la suite O_{t+1}..O_T en partant de l'état i à t
 
  Initialisation : β_T(i) = 1 pour tout i
  Récurrence    : β_t(i) = Σ_j a_ij · B_j(O_{t+1}) · β_{t+1}(j)
  Finalisation  : P(O|λ) = Σ_i π_i · B_i(O_1) · β_1(i)

  Matrice β (T×N) — probabilités backward :
     t       β_t(Normal)    β_t(Intrusion)
     1        0.00033770        0.00033388
     2        0.00062594        0.00061890
     3        0.00115753        0.00117115
     4        0.00592001        0.00600497
     5        0.02243704        0.02218321
     6        0.04158832        0.04111785
     7        0.07708580        0.07621885
     8        0.14251430        0.14455971
     9        0.54011837        0.53402394
    10        1.00000000        1.00000000

  P(O|λ) via backward = 0.0001811782
  Cohérence Forward = Backward : True


#  PARTIE 5 : Algorithme de Viterbi δ, ψ


In [7]:
print("\n" + "=" * 70)
print("PARTIE 5 — Algorithme de Viterbi δ_t(i), ψ_t(i)")
print("=" * 70)
print("""
  Objectif : Trouver la séquence d'états cachés X* la plus probable
 
  Définition : δ_t(i) = max_{X_1..X_{t-1}} P(X_1..X_{t-1}, X_t=i, O_1..O_t | λ)
 
  Initialisation : δ_1(i) = π_i · B_i(O_1),  ψ_1(i) = 0
  Récurrence    : δ_t(j) = max_i[δ_{t-1}(i) · a_ij] · B_j(O_t)
                  ψ_t(j) = argmax_i[δ_{t-1}(i) · a_ij]
  Finalisation  : P* = max_i δ_T(i),  X*_T = argmax_i δ_T(i)
  Backtracking  : X*_t = ψ_{t+1}(X*_{t+1})
""")
 
def viterbi(obs_seq, A, B, pi):
    """Algorithme de Viterbi."""
    T = len(obs_seq)
    N = A.shape[0]
    delta = np.zeros((T, N))
    psi   = np.zeros((T, N), dtype=int)
 
    # Initialisation
    delta[0] = pi * B[:, obs_seq[0]]
    psi[0]   = 0
 
    # Récurrence
    for t in range(1, T):
        for j in range(N):
            scores = delta[t-1] * A[:, j]
            psi[t, j]   = np.argmax(scores)
            delta[t, j] = np.max(scores) * B[j, obs_seq[t]]
 
    # Finalisation
    prob_star = np.max(delta[-1])
    seq_star  = np.zeros(T, dtype=int)
    seq_star[-1] = np.argmax(delta[-1])
 
    # Backtracking
    for t in range(T-2, -1, -1):
        seq_star[t] = psi[t+1, seq_star[t+1]]
 
    return delta, psi, seq_star, prob_star
 
delta, psi, seq_viterbi, prob_viterbi = viterbi(obs_seq_test, A, B, pi)
 
print(f"  Matrice δ (T×N) — probabilités maximales :")
print(f"  {'t':>4s}  {'δ_t(Normal)':>16s}  {'δ_t(Intrusion)':>16s}  {'ψ_t(N)':>8s}  {'ψ_t(I)':>8s}")
for t in range(len(obs_seq_test)):
    print(f"  {t+1:>4d}  {delta[t,0]:>16.8f}  {delta[t,1]:>16.8f}  "
          f"{psi[t,0]:>8d}  {psi[t,1]:>8d}")
 
print(f"\n  Séquence d'états cachés la plus probable (Viterbi) :")
print(f"    X* = {[ETATS_CACHES[e] for e in seq_viterbi]}")
print(f"    P* = {prob_viterbi:.10f}")
print(f"\n  États réels (NSL-KDD) :")
print(f"    X  = {[ETATS_CACHES[e] for e in etats_reels]}")
 
# Précision de Viterbi
accuracy = np.mean(seq_viterbi == etats_reels)
print(f"\n  Précision Viterbi sur les 10 premières connexions : {accuracy*100:.1f}%")
 
# Évaluation sur un ensemble plus grand
obs_test_100  = df['observation'].values[:500]
etats_vrais_100 = df['etat_cache'].values[:500]
_, _, seq_vit_100, _ = viterbi(obs_test_100, A, B, pi)
acc_100 = np.mean(seq_vit_100 == etats_vrais_100)
print(f"  Précision Viterbi sur 500 connexions : {acc_100*100:.2f}%")


PARTIE 5 — Algorithme de Viterbi δ_t(i), ψ_t(i)

  Objectif : Trouver la séquence d'états cachés X* la plus probable
 
  Définition : δ_t(i) = max_{X_1..X_{t-1}} P(X_1..X_{t-1}, X_t=i, O_1..O_t | λ)
 
  Initialisation : δ_1(i) = π_i · B_i(O_1),  ψ_1(i) = 0
  Récurrence    : δ_t(j) = max_i[δ_{t-1}(i) · a_ij] · B_j(O_t)
                  ψ_t(j) = argmax_i[δ_{t-1}(i) · a_ij]
  Finalisation  : P* = max_i δ_T(i),  X*_T = argmax_i δ_T(i)
  Backtracking  : X*_t = ψ_{t+1}(X*_{t+1})

  Matrice δ (T×N) — probabilités maximales :
     t       δ_t(Normal)    δ_t(Intrusion)    ψ_t(N)    ψ_t(I)
     1        0.48237500        0.05475000         0         0
     2        0.23431444        0.02622516         0         0
     3        0.11381862        0.01273891         0         0
     4        0.00202010        0.01994836         0         0
     5        0.00000000        0.00540000         1         1
     6        0.00258594        0.00029779         1         1
     7        0.00125612        0

#  PARTIE 6 : Algorithme de Baum-Welch EM


In [8]:
print("\n" + "=" * 70)
print("PARTIE 6 — Algorithme de Baum-Welch (EM)")
print("=" * 70)
print("""
  Objectif : Apprendre λ=(A,B,π) depuis les observations non étiquetées
             argmax_λ P(O|λ)
 
  Étape E : Calculer γ_t(i) et ξ_t(i,j)
    γ_t(i) = α_t(i)·β_t(i) / Σ_k α_t(k)·β_t(k)
    ξ_t(i,j) = α_t(i)·a_ij·B_j(O_{t+1})·β_{t+1}(j) / P(O|λ)
 
  Étape M : Mettre à jour les paramètres
    A_ij = Σ_{t=1}^{T-1} ξ_t(i,j) / Σ_{t=1}^{T-1} γ_t(i)
    B_j(k) = Σ_{t:O_t=k} γ_t(j)  / Σ_t γ_t(j)
    π_i = γ_1(i)
""")
 
def baum_welch(obs_seq, N, M, n_iter=20):
    """
    Algorithme de Baum-Welch complet.
    Retourne A, B, pi optimisés + historique de log-vraisemblance.
    """
    T = len(obs_seq)
 
    # ── Initialisation aléatoire (avec bruit) ──
    A_bw = np.array([[0.9, 0.1], [0.4, 0.6]])
    B_bw = np.array([[0.7, 0.25, 0.05], [0.1, 0.3, 0.6]])
    pi_bw = np.array([0.9, 0.1])
 
    log_vrais_hist = []
    A_hist, B_hist, pi_hist = [], [], []
 
    for iteration in range(n_iter):
        # ── ÉTAPE E ──
        alpha_bw, prob = forward(obs_seq, A_bw, B_bw, pi_bw)
        beta_bw, _     = backward(obs_seq, A_bw, B_bw, pi_bw)
 
        if prob < 1e-300:
            break
 
        # γ_t(i) = P(X_t=i | O, λ)
        gamma = alpha_bw * beta_bw
        gamma /= gamma.sum(axis=1, keepdims=True)
 
        # ξ_t(i,j) = P(X_t=i, X_{t+1}=j | O, λ)
        xi = np.zeros((T-1, N, N))
        for t in range(T-1):
            for i in range(N):
                for j in range(N):
                    xi[t, i, j] = (alpha_bw[t, i] * A_bw[i, j] *
                                   B_bw[j, obs_seq[t+1]] * beta_bw[t+1, j])
            xi[t] /= xi[t].sum() + 1e-300
 
        # ── ÉTAPE M ──
        # Mise à jour A
        A_new = xi.sum(axis=0) / gamma[:-1].sum(axis=0, keepdims=True).T
        A_new /= A_new.sum(axis=1, keepdims=True)
 
        # Mise à jour B
        B_new = np.zeros((N, M))
        for k in range(M):
            mask = (obs_seq == k)
            B_new[:, k] = gamma[mask].sum(axis=0)
        B_new /= B_new.sum(axis=1, keepdims=True)
 
        # Mise à jour π
        pi_new = gamma[0]
 
        # Log-vraisemblance
        log_p = np.log(prob + 1e-300)
        log_vrais_hist.append(log_p)
        A_hist.append(A_new.copy())
        B_hist.append(B_new.copy())
        pi_hist.append(pi_new.copy())
 
        if iteration > 0 and abs(log_p - log_vrais_hist[-2]) < 1e-6:
            print(f"    Convergence atteinte à l'itération {iteration+1}")
            break
 
        A_bw, B_bw, pi_bw = A_new, B_new, pi_new
 
    return A_bw, B_bw, pi_bw, log_vrais_hist, A_hist, B_hist, pi_hist
 
# Entraîner sur les observations (non étiquetées — apprentissage non supervisé)
obs_train = df['observation'].values[:300]
 
print(f"  Entraînement Baum-Welch sur {len(obs_train)} observations NSL-KDD...")
print(f"  Initialisation : A=[[0.9,0.1],[0.4,0.6]], B=[[0.7,0.25,0.05],[0.1,0.3,0.6]]")
print(f"  Distribution initiale : π=[0.9, 0.1]")
print()
 
A_bw, B_bw, pi_bw, log_hist, A_hist, B_hist, pi_hist = baum_welch(
    obs_train, N=2, M=3, n_iter=20
)
 
print(f"\n  Résultats après Baum-Welch :")
print(f"\n  A (Matrice de transition mise à jour) :")
print(f"    [[{A_bw[0,0]:.4f}  {A_bw[0,1]:.4f}]")
print(f"     [{A_bw[1,0]:.4f}  {A_bw[1,1]:.4f}]]")
print(f"\n  B (Matrice d'émission mise à jour) :")
print(f"    [[{B_bw[0,0]:.4f}  {B_bw[0,1]:.4f}  {B_bw[0,2]:.4f}]")
print(f"     [{B_bw[1,0]:.4f}  {B_bw[1,1]:.4f}  {B_bw[1,2]:.4f}]]")
print(f"\n  π (Distribution initiale mise à jour) :")
print(f"    [{pi_bw[0]:.4f}  {pi_bw[1]:.4f}]")
 
# Afficher les premières itérations (comme le cours pages 35-42)
print(f"\n  Évolution des paramètres par itération (comme le cours p.35-42) :")
for it in range(min(4, len(A_hist))):
    print(f"\n  ── Itération {it+1} ──")
    print(f"    A : [[{A_hist[it][0,0]:.3f} {A_hist[it][0,1]:.3f}] "
          f"[{A_hist[it][1,0]:.3f} {A_hist[it][1,1]:.3f}]]")
    print(f"    B : [[{B_hist[it][0,0]:.3f} {B_hist[it][0,1]:.3f} {B_hist[it][0,2]:.3f}] "
          f"[{B_hist[it][1,0]:.3f} {B_hist[it][1,1]:.3f} {B_hist[it][1,2]:.3f}]]")
    print(f"    π : [{pi_hist[it][0]:.3f}  {pi_hist[it][1]:.3f}]")
    print(f"    log P(O|λ) = {log_hist[it]:.6f}")


PARTIE 6 — Algorithme de Baum-Welch (EM)

  Objectif : Apprendre λ=(A,B,π) depuis les observations non étiquetées
             argmax_λ P(O|λ)
 
  Étape E : Calculer γ_t(i) et ξ_t(i,j)
    γ_t(i) = α_t(i)·β_t(i) / Σ_k α_t(k)·β_t(k)
    ξ_t(i,j) = α_t(i)·a_ij·B_j(O_{t+1})·β_{t+1}(j) / P(O|λ)
 
  Étape M : Mettre à jour les paramètres
    A_ij = Σ_{t=1}^{T-1} ξ_t(i,j) / Σ_{t=1}^{T-1} γ_t(i)
    B_j(k) = Σ_{t:O_t=k} γ_t(j)  / Σ_t γ_t(j)
    π_i = γ_1(i)

  Entraînement Baum-Welch sur 300 observations NSL-KDD...
  Initialisation : A=[[0.9,0.1],[0.4,0.6]], B=[[0.7,0.25,0.05],[0.1,0.3,0.6]]
  Distribution initiale : π=[0.9, 0.1]


  Résultats après Baum-Welch :

  A (Matrice de transition mise à jour) :
    [[0.8276  0.1724]
     [0.6929  0.3071]]

  B (Matrice d'émission mise à jour) :
    [[0.6854  0.1946  0.1200]
     [0.2554  0.0873  0.6573]]

  π (Distribution initiale mise à jour) :
    [1.0000  0.0000]

  Évolution des paramètres par itération (comme le cours p.35-42) :

  ── Itérati

#  PARTIE 7 : Évaluation finale du modèle HMM appris


In [9]:
print("\n" + "=" * 70)
print("PARTIE 7 — Évaluation du Modèle HMM Appris")
print("=" * 70)
 
obs_eval  = df['observation'].values[300:800]
etats_eval = df['etat_cache'].values[300:800]
 
_, _, seq_vit_bw, prob_bw = viterbi(obs_eval, A_bw, B_bw, pi_bw)
acc_bw = np.mean(seq_vit_bw == etats_eval)
 
_, prob_orig = forward(obs_eval, A, B, pi)
_, _, seq_vit_orig, _ = viterbi(obs_eval, A, B, pi)
acc_orig = np.mean(seq_vit_orig == etats_eval)
 
print(f"\n  Comparaison avant/après Baum-Welch sur 500 connexions :")
print(f"    {'Modèle':30s}  {'Précision':>10s}  {'log P(O|λ)':>12s}")
print(f"    {'─'*56}")
print(f"    {'HMM initial (depuis NSL-KDD)':30s}  {acc_orig*100:>9.2f}%  {np.log(prob_orig+1e-300):>12.4f}")
print(f"    {'HMM Baum-Welch (appris)':30s}  {acc_bw*100:>9.2f}%  {np.log(prob_bw+1e-300):>12.4f}")
 
print(f"\n  Vraisemblance P(O|λ) comparée :")
alpha_eval, prob_eval_bw = forward(obs_eval[:20], A_bw, B_bw, pi_bw)
_, prob_eval_orig = forward(obs_eval[:20], A, B, pi)
print(f"    P(O|λ_initial)   = {prob_eval_orig:.6e}")
print(f"    P(O|λ_baum-welch) = {prob_eval_bw:.6e}")
 
print("\n" + "=" * 70)
print("  ✅ CHAPITRE 3 COMPLET — Tous les algorithmes implémentés")
print("=" * 70)


PARTIE 7 — Évaluation du Modèle HMM Appris

  Comparaison avant/après Baum-Welch sur 500 connexions :
    Modèle                           Précision    log P(O|λ)
    ────────────────────────────────────────────────────────
    HMM initial (depuis NSL-KDD)        92.80%     -507.5346
    HMM Baum-Welch (appris)             62.40%     -614.2265

  Vraisemblance P(O|λ) comparée :
    P(O|λ_initial)   = 4.670115e-08
    P(O|λ_baum-welch) = 1.459004e-07

  ✅ CHAPITRE 3 COMPLET — Tous les algorithmes implémentés


#  PARTIE 8 : VISUALISATIONS — 3 figures séparées


In [11]:
COUL_ETAT = {0:"#1D9E75", 1:"#D85A30"}
COUL_OBS  = {0:"#378ADD", 1:"#BA7517", 2:"#534AB7"}
SAVE = r'C:\Users\Fujitsu\Desktop\Data science\S3\ML probabilistique\project of module\projet_cybersecurity'
 
# ── FIGURE 1 : Structure HMM + Matrices A, B ──
fig1, axes = plt.subplots(1, 3, figsize=(22, 7))
fig1.suptitle("Chapitre 3 — DT-HMM | Fig.1 : Structure & Paramètres λ=(A,B,π)\n"
              "Dataset : NSL-KDD — Détection d'Intrusions",
              fontsize=14, fontweight="bold", y=1.02)
 
# 1A : Heatmap matrice A
ax = axes[0]
im1 = ax.imshow(A, cmap="Greens", vmin=0, vmax=1)
for i in range(N):
    for j in range(N):
        ax.text(j, i, f"{A[i,j]:.4f}", ha="center", va="center",
                fontsize=13, fontweight="bold",
                color="white" if A[i,j]>0.5 else "black")
ax.set_xticks(range(N)); ax.set_yticks(range(N))
ax.set_xticklabels([ETATS_CACHES[i] for i in range(N)], fontsize=12)
ax.set_yticklabels([ETATS_CACHES[i] for i in range(N)], fontsize=12)
ax.set_title("Matrice de Transition A\na_ij = P(X_{t+1}=j | X_t=i)", fontsize=13, fontweight="bold")
plt.colorbar(im1, ax=ax, fraction=0.046)
ax.set_xlabel("État suivant (t+1)", fontsize=11)
ax.set_ylabel("État actuel (t)", fontsize=11)
 
# 1B : Heatmap matrice B
ax = axes[1]
im2 = ax.imshow(B, cmap="Blues", vmin=0, vmax=1)
for i in range(N):
    for k in range(M):
        ax.text(k, i, f"{B[i,k]:.4f}", ha="center", va="center",
                fontsize=13, fontweight="bold",
                color="white" if B[i,k]>0.5 else "black")
ax.set_xticks(range(M)); ax.set_yticks(range(N))
ax.set_xticklabels([OBS_LABELS[k] for k in range(M)], fontsize=10, rotation=15)
ax.set_yticklabels([ETATS_CACHES[i] for i in range(N)], fontsize=12)
ax.set_title("Matrice d'Émission B\nb_ik = P(O_t=k | X_t=i)", fontsize=13, fontweight="bold")
plt.colorbar(im2, ax=ax, fraction=0.046)
ax.set_xlabel("Observation", fontsize=11)
ax.set_ylabel("État caché", fontsize=11)
 
# 1C : Distribution des observations par état
ax = axes[2]
x = np.arange(M); w = 0.35
for i in range(N):
    ax.bar(x + i*w - w/2, B[i]*100, w,
           label=f"État {ETATS_CACHES[i]}",
           color=COUL_ETAT[i], alpha=0.85,
           edgecolor='white', linewidth=1.2)
ax.set_xticks(x)
ax.set_xticklabels([OBS_LABELS[k] for k in range(M)], fontsize=11)
ax.set_ylabel("Probabilité d'émission (%)", fontsize=12)
ax.set_title("Probabilités d'Émission B\n(par état caché)", fontsize=13, fontweight="bold")
ax.legend(fontsize=11, framealpha=0.9)
ax.grid(axis='y', alpha=0.3)
# Distribution initiale π
ax.text(0.02, 0.98,
        f"Distribution initiale π :\n  π(Normal)    = {pi[0]:.3f}\n  π(Intrusion) = {pi[1]:.3f}",
        transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round', fc='#fffbe6', alpha=0.9))
for i in range(N):
    for k in range(M):
        val = B[i,k]*100
        ax.text(k + i*w - w/2, val+0.5, f"{val:.1f}%",
                ha='center', va='bottom', fontsize=9)
 
plt.tight_layout()
plt.savefig(f'{SAVE}/ch3_fig1_parametres.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.close()
print("\n✅ Figure 1 sauvegardée")


✅ Figure 1 sauvegardée


In [12]:
fig2, axes2 = plt.subplots(1, 3, figsize=(22, 7))
fig2.suptitle("Chapitre 3 — DT-HMM | Fig.2 : Algorithmes Forward, Backward & Viterbi\n"
              "Dataset : NSL-KDD — Séquence de 10 connexions",
              fontsize=14, fontweight="bold", y=1.02)
 
T_plot = len(obs_seq_test)
t_axis = np.arange(1, T_plot+1)
 
# 2A : Forward α
ax = axes2[0]
for i in range(N):
    ax.plot(t_axis, alpha[:, i], 'o-', color=COUL_ETAT[i],
            lw=2.5, markersize=8, label=f"α_t({ETATS_CACHES[i]})")
    for t in range(T_plot):
        ax.annotate(f"{alpha[t,i]:.4f}", (t+1, alpha[t,i]),
                    textcoords="offset points", xytext=(0,8),
                    ha='center', fontsize=7, color=COUL_ETAT[i])
ax.set_xlabel("Instant t", fontsize=12)
ax.set_ylabel("α_t(i) = P(O_1..O_t, X_t=i | λ)", fontsize=11)
ax.set_title(f"Algorithme de Forward α\nP(O|λ) = {prob_forward:.2e}", fontsize=13, fontweight="bold")
ax.legend(fontsize=11); ax.grid(alpha=0.3)
ax.set_xticks(t_axis)
 
# 2B : Backward β
ax = axes2[1]
for i in range(N):
    ax.plot(t_axis, beta[:, i], 's--', color=COUL_ETAT[i],
            lw=2.5, markersize=8, label=f"β_t({ETATS_CACHES[i]})")
    for t in range(T_plot):
        ax.annotate(f"{beta[t,i]:.4f}", (t+1, beta[t,i]),
                    textcoords="offset points", xytext=(0,8),
                    ha='center', fontsize=7, color=COUL_ETAT[i])
ax.set_xlabel("Instant t", fontsize=12)
ax.set_ylabel("β_t(i) = P(O_{t+1}..O_T | X_t=i, λ)", fontsize=11)
ax.set_title(f"Algorithme de Backward β\nP(O|λ) = {prob_backward:.2e}", fontsize=13, fontweight="bold")
ax.legend(fontsize=11); ax.grid(alpha=0.3)
ax.set_xticks(t_axis)
 
# 2C : Viterbi — séquence d'états + observations
ax = axes2[2]
obs_colors = [COUL_OBS[o] for o in obs_seq_test]
vit_colors = [COUL_ETAT[e] for e in seq_viterbi]
real_colors = [COUL_ETAT[e] for e in etats_reels]
ax.scatter(t_axis, seq_viterbi + 0.05, s=180, c=vit_colors,
           marker='D', zorder=4, label="Viterbi X*")
ax.scatter(t_axis, etats_reels - 0.05, s=180, c=real_colors,
           marker='o', zorder=3, alpha=0.6, label="État réel X")
ax.step(t_axis, seq_viterbi, where='mid', color='#185FA5', lw=1.5, alpha=0.6)
ax.step(t_axis, etats_reels, where='mid', color='gray', lw=1.5, ls='--', alpha=0.6)
# Observations en bas
for t in range(T_plot):
    ax.text(t+1, -0.35, OBS_LABELS[obs_seq_test[t]][:4],
            ha='center', fontsize=8, color=COUL_OBS[obs_seq_test[t]],
            fontweight='bold')
ax.set_yticks([0, 1])
ax.set_yticklabels([ETATS_CACHES[i] for i in range(N)], fontsize=12)
ax.set_xlabel("Instant t  (observations en bas)", fontsize=12)
ax.set_title(f"Algorithme de Viterbi\nPrécision = {accuracy*100:.1f}%,  P* = {prob_viterbi:.2e}",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11, loc='upper right')
ax.grid(axis='y', alpha=0.3)
ax.set_xticks(t_axis)
ax.set_ylim(-0.5, 1.5)
patches = [mpatches.Patch(color=COUL_OBS[k], label=OBS_LABELS[k]) for k in range(M)]
ax.legend(handles=patches, fontsize=8, loc='lower right', title="Observations")
 
plt.tight_layout()
plt.savefig(f'{SAVE}/ch3_fig2_algorithmes.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.close()
print("✅ Figure 2 sauvegardée")

✅ Figure 2 sauvegardée


In [13]:
fig3, axes3 = plt.subplots(1, 3, figsize=(22, 7))
fig3.suptitle("Chapitre 3 — DT-HMM | Fig.3 : Apprentissage Baum-Welch (EM)\n"
              "Dataset : NSL-KDD — Convergence & Mise à Jour des Paramètres",
              fontsize=14, fontweight="bold", y=1.02)
 
# 3A : Log-vraisemblance au fil des itérations
ax = axes3[0]
ax.plot(range(1, len(log_hist)+1), log_hist, 'o-',
        color="#185FA5", lw=2.5, markersize=8)
ax.fill_between(range(1, len(log_hist)+1), log_hist,
                alpha=0.15, color="#185FA5")
ax.set_xlabel("Itération EM", fontsize=12)
ax.set_ylabel("log P(O|λ)", fontsize=12)
ax.set_title("Convergence Baum-Welch\nlog P(O|λ) par itération", fontsize=13, fontweight="bold")
ax.grid(alpha=0.3)
ax.text(0.98, 0.05,
        f"Convergence en\n{len(log_hist)} itérations",
        transform=ax.transAxes, ha='right', fontsize=11,
        bbox=dict(boxstyle='round', fc='#e8f5e9', alpha=0.9))
for it, val in enumerate(log_hist):
    ax.annotate(f"{val:.3f}", (it+1, val),
                textcoords="offset points", xytext=(0, 8),
                ha='center', fontsize=8)
 
# 3B : Évolution de A au fil des itérations
ax = axes3[1]
iters = range(1, len(A_hist)+1)
ax.plot(iters, [A_hist[it][0,0] for it in range(len(A_hist))],
        'o-', color=COUL_ETAT[0], lw=2, label="A[Normal→Normal]")
ax.plot(iters, [A_hist[it][0,1] for it in range(len(A_hist))],
        's--', color=COUL_ETAT[0], lw=2, alpha=0.7, label="A[Normal→Intrusion]")
ax.plot(iters, [A_hist[it][1,0] for it in range(len(A_hist))],
        '^-', color=COUL_ETAT[1], lw=2, label="A[Intrusion→Normal]")
ax.plot(iters, [A_hist[it][1,1] for it in range(len(A_hist))],
        'D--', color=COUL_ETAT[1], lw=2, alpha=0.7, label="A[Intrusion→Intrusion]")
ax.set_xlabel("Itération EM", fontsize=12)
ax.set_ylabel("Probabilité de transition a_ij", fontsize=12)
ax.set_title("Évolution de A (Matrice Transition)\nÉtape M — Mise à jour",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9, framealpha=0.9)
ax.grid(alpha=0.3)
ax.set_ylim(0, 1)
 
# 3C : Comparaison B initial vs B Baum-Welch
ax = axes3[2]
x = np.arange(M); w = 0.2
for i in range(N):
    ax.bar(x + i*w*2 - w, B[i]*100, w,
           label=f"{ETATS_CACHES[i]} initial",
           color=COUL_ETAT[i], alpha=0.85, edgecolor='white')
    ax.bar(x + i*w*2, B_bw[i]*100, w,
           label=f"{ETATS_CACHES[i]} Baum-Welch",
           color=COUL_ETAT[i], alpha=0.45,
           edgecolor=COUL_ETAT[i], linewidth=2, hatch='//')
ax.set_xticks(x + w/2)
ax.set_xticklabels([OBS_LABELS[k] for k in range(M)], fontsize=11)
ax.set_ylabel("Probabilité d'émission (%)", fontsize=12)
ax.set_title("Matrice d'Émission B\nAvant vs Après Baum-Welch",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9, framealpha=0.9, ncol=2)
ax.grid(axis='y', alpha=0.3)
ax.text(0.5, 0.97,
        f"Précision Viterbi :\n"
        f"  Initial : {acc_orig*100:.1f}%\n"
        f"  Baum-Welch : {acc_bw*100:.1f}%",
        transform=ax.transAxes, ha='center', va='top', fontsize=11,
        bbox=dict(boxstyle='round', fc='#fff9e6', alpha=0.95))
 
plt.tight_layout()
plt.savefig(f'{SAVE}/ch3_fig3_baum_welch.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.close()
print("✅ Figure 3 sauvegardée")


✅ Figure 3 sauvegardée
